# ktrans — 괴테 『젊은 베르터의 고뇌』 한국어 번역·비평 Linked Data

**Modelling Korean Translations of German Literature as Linked Open Data**

UeDeKo 번역서지 318 레코드와 번역비평 텍스트를 LRMoo·CIDOC CRM 기반 RDF로 편찬하고
SPARQL로 검증한다. 설계 원칙은 표준을 최대한 그대로 쓰는 것이며, 자체 정의는
클래스 1개(`ktrans:Translation`)와 프로퍼티 3개(`sourceType`·`viaType`·`hasPolarity`)뿐이다.

| | |
|---|---|
| 데이터 출처 | UeDeKo (MediaWiki 1.35.1) |
| 골격 | LRMoo v1.1.1 |
| 상위 모델 | CIDOC CRM v7.1.3 |
| 설계 원리 | MiMoText |
| 구현 | rdflib / Turtle |

메모(`crm:P3_has_note`)와 비평 진술 요지(`rdfs:label`)는 한국어(@ko)와 영어(@en)가 함께 있다. 질의에서 `lang()`으로 언어를 고른다.
Notes and statement summaries exist in Korean (@ko) and English (@en); queries pick a language with `lang()`.

## 0. 준비 — 그래프 로드

In [ ]:
import warnings, matplotlib
import matplotlib.font_manager as fm
warnings.filterwarnings("ignore")

# 설치된 한글 폰트를 자동으로 찾는다 (윈도우·맥·리눅스 공통)
_cands = ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic",
          "Noto Sans CJK KR", "Gulim", "Batang"]
_have = {f.name for f in fm.fontManager.ttflist}
KFONT = next((c for c in _cands if c in _have), None)
if KFONT:
    matplotlib.rcParams["font.family"] = KFONT
    print(f"한글 폰트: {KFONT}")
else:
    print("한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")
matplotlib.rcParams["axes.unicode_minus"] = False

import pandas as pd
import matplotlib.pyplot as plt
from rdflib import Graph

g = Graph()
g.parse("ktrans-schema.ttl", format="turtle")
g.parse("ktrans-data.ttl", format="turtle")
print(f"트리플 {len(g):,}개")

In [ ]:
PREFIX = """
PREFIX ktrans: <https://ktrans.aks.ac.kr/ontology#>
PREFIX kt:     <https://ktrans.aks.ac.kr/id/>
PREFIX lrmoo:  <http://iflastandards.info/ns/lrm/lrmoo/>
PREFIX crm:    <http://www.cidoc-crm.org/cidoc-crm/7.1.3/>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
"""

def q(sparql):
    """SPARQL 실행 -> DataFrame"""
    res = g.query(PREFIX + sparql)
    cols = [str(v) for v in res.vars]
    return pd.DataFrame([[str(c) if c is not None else "" for c in row]
                         for row in res], columns=cols)

## 1. 데이터 규모

레코드 318건이 작품 271건으로 정리되었다. 연재·분책·중복등재를 통합한 결과이며,
통합 판단은 번역본마다 메모의 (편찬) 문장에 근거와 함께 남아 있다.

In [ ]:
q("""
SELECT (COUNT(DISTINCT ?t) AS ?번역본)
       (COUNT(DISTINCT ?ev) AS ?번역사건)
       (COUNT(DISTINCT ?p) AS ?인물)
       (COUNT(DISTINCT ?cs) AS ?비평진술)
WHERE {
  { ?t a ktrans:Translation } UNION
  { ?ev a lrmoo:F28_Expression_Creation } UNION
  { ?p a crm:E21_Person } UNION
  { ?cs crm:P2_has_type kt:type-criticism }
}
""")

## Q1. 저자 표기 26종이 하나의 인물로 수렴한다

평면 데이터베이스에서 이 26개는 서로 다른 문자열이다. 괴테의 작품을 한 번에 셀 수 없다.
`F1 Work`에 정규형 하나를 두고 271개 작품을 연결하면 표기가 어떻든 하나의 저작으로 묶인다.

> 저자명 필드에는 27종의 값이 나타나며, 그중 하나(`베르테르`)는 저자가 아니라 작중인물이다.
> 평면 DB에서는 걸러지지 않지만, F1에 연결하는 순간 이 레코드만 연결되지 않아 이상 사례가 드러난다.

In [ ]:
df = q("""
SELECT ?표기 WHERE { kt:P-goethe skos:altLabel ?표기 } ORDER BY ?표기
""")
print(f"표기 이형 {len(df)}종\n")
print(" / ".join(df["표기"]))

## Q2. 번역의 저본과 그 확실성

`F28 Expression Creation` 사건에 저본(`P16`)·경유유형·확실성을 걸었다.
판본이 특정되지 않아도 "역자가 저본을 밝혔다"는 상태가 데이터에 남는다.

트리어 피드백은 직역/중역을 `R76.1`로 관계에 붙이라고 지적했다. 그 취지를 받아들이되
순수 RDF에서 재화 없이 표현되지 않으므로, LRMoo가 본래 제공하는 F28 사건으로 구현했다.
LRMoo 자신이 `R2`를 F28 경로의 축약으로 정의하므로 축약과 사건을 함께 쓰는 것은 표준 패턴이다.

In [ ]:
q("""
SELECT ?연도 ?번역본 ?역자 ?저본 ?경유 ?확실성
WHERE {
  ?ev a lrmoo:F28_Expression_Creation ;
      lrmoo:R17_created ?t ; crm:P2_has_type ?c .
  ?c skos:inScheme kt:scheme-certainty ; rdfs:label ?확실성 .
  ?t rdfs:label ?번역본 ; crm:P4_has_time-span/crm:P82_at_some_time_within ?연도 .
  OPTIONAL { ?ev crm:P14_carried_out_by ?p . ?p rdfs:label ?역자 . FILTER(lang(?역자)="ko") }
  OPTIONAL { ?ev crm:P16_used_specific_object ?s . ?s rdfs:label ?저본 . FILTER(lang(?저본)="ko") }
  OPTIONAL { ?ev crm:P2_has_type ?v . ?v skos:inScheme kt:scheme-viaType ;
             rdfs:label ?경유 . FILTER(lang(?경유)="ko") }
  FILTER(lang(?번역본)="ko" && lang(?확실성)="ko")
} ORDER BY ?연도
""")

## Q3. 중역 경로 — 두 갈래의 파생 연쇄

1923년 같은 해에 나온 두 번역이 서로 다른 언어를 거쳤다.
김영보는 일본어를, 백화(양건식)는 중국어를 경유했다. `R76` 연쇄로 표현된다.

In [ ]:
chain = q("""
SELECT ?번역본 ?연도 ?직접저본 ?그원천
WHERE {
  ?t ktrans:viaType kt:via-relay ; rdfs:label ?번역본 ;
     crm:P4_has_time-span/crm:P82_at_some_time_within ?연도 .
  ?ev lrmoo:R17_created ?t ; crm:P16_used_specific_object ?s .
  ?s rdfs:label ?직접저본 ; lrmoo:R76_is_derivative_of ?o .
  ?o rdfs:label ?그원천 .
  FILTER(lang(?번역본)="ko" && lang(?직접저본)="ko" && lang(?그원천)="ko")
} ORDER BY ?연도
""")
chain

In [ ]:
import networkx as nx

def short(s):
    return s.split("(")[0].strip()[:22]

G = nx.DiGraph()
pos, labels = {}, {}
G.add_node("DE"); pos["DE"] = (0, 0); labels["DE"] = "독일어 원문\n(1774)"

mid = {}
for i, r in chain.iterrows():
    m = short(r["직접저본"])
    if m not in mid:
        mid[m] = f"M{len(mid)}"
        G.add_node(mid[m]); pos[mid[m]] = (1, 0.6 - 1.2*len(mid[m])*0 - 1.2*(len(mid)-1))
        labels[mid[m]] = m.replace(" ", "\n", 1)
        G.add_edge("DE", mid[m])
for i, r in chain.iterrows():
    k = f"K{i}"
    m = mid[short(r["직접저본"])]
    G.add_node(k); pos[k] = (2, pos[m][1]); labels[k] = f'{short(r["번역본"])}\n({r["연도"]})'
    G.add_edge(m, k)

fig, ax = plt.subplots(figsize=(11, 4.2))
nx.draw_networkx_nodes(G, pos, node_size=5200, node_color="#F7EEF1",
                       edgecolors="#6D2E46", linewidths=1.6, ax=ax)
nx.draw_networkx_edges(G, pos, arrowsize=22, width=1.8, edge_color="#6D2E46",
                       node_size=5200, ax=ax)
nx.draw_networkx_labels(G, pos, labels, font_size=9,
                        font_family=KFONT, ax=ax)
for (u, v) in G.edges():
    x = (pos[u][0] + pos[v][0]) / 2; y = (pos[u][1] + pos[v][1]) / 2
    ax.text(x, y + 0.08, "R76", fontsize=8, color="#6D2E46", ha="center")
ax.set_title("중역 연쇄 — lrmoo:R76_is_derivative_of", fontsize=12, color="#6D2E46", pad=14)
ax.axis("off"); plt.tight_layout(); plt.show()

## Q4. Leiden 역어의 시대별 변천

갈라짐 → 모임 → 다시 갈라짐. 1920년대에 네 갈래로 갈렸다가 '슬픔'으로 수렴하고,
2010년대에 고통·고뇌·괴로움으로 다시 분화한다.
`F12 Nomen`과 `R56_has_related_form`으로 원제와 각 역어를 이어 추적한다.

In [ ]:
leiden = q("""
SELECT ?연도 ?역어 (COUNT(?n) AS ?건수)
WHERE {
  ?n lrmoo:R56_has_related_form kt:nomen-W01-de ;
     crm:P3_has_note ?메모 ;
     crm:P4_has_time-span/crm:P82_at_some_time_within ?연도 .
  BIND(STRBEFORE(STRAFTER(STR(?메모), "Leiden을 '"), "'") AS ?역어)
  FILTER(?역어 != "")
} GROUP BY ?연도 ?역어 ORDER BY ?연도
""")
leiden["연도"] = leiden["연도"].astype(int)
leiden["건수"] = leiden["건수"].astype(int)
leiden.head(12)

In [ ]:
order = ["悲歌", "悲惱", "서름", "슬픔", "번민", "고통", "고뇌", "괴로움"]
present = [t for t in order if t in set(leiden["역어"])]
ymap = {t: i for i, t in enumerate(present)}
colors = plt.cm.viridis([i/max(len(present)-1,1) for i in range(len(present))])

fig, ax = plt.subplots(figsize=(12, 4.4))
for t in present:
    d = leiden[leiden["역어"] == t]
    ax.scatter(d["연도"], [ymap[t]]*len(d), s=d["건수"]*55,
               color=colors[ymap[t]], alpha=.8, edgecolors="white", linewidths=1,
               zorder=3, label=t)
ax.set_yticks(range(len(present)))
ax.set_yticklabels(present, fontsize=11,
                   fontfamily=KFONT)
ax.set_xlabel("출판 연도"); ax.grid(axis="x", alpha=.25, zorder=0)
ax.set_title("'Leiden' 역어의 시대별 변천  —  원 크기 = 해당 연도 판본 수",
             fontsize=12, color="#6D2E46", pad=12)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Q5. 번역본별 비평 진술

줄글 비평이 **무엇을·어떤 기준으로·어떤 극성으로·어디에 근거해**의 네 갈래로 분해되어
질의된다. 모든 진술은 `P16`으로 원문 문단(para_id)에 연결되므로 되짚을 수 있다.

In [ ]:
stmt = q("""
SELECT ?번역본 ?차원 ?값 ?극성 ?주체 ?근거
WHERE {
  ?cs crm:P2_has_type kt:type-criticism ;
      crm:P140_assigned_attribute_to ?t ;
      crm:P14_carried_out_by ?p ;
      crm:P16_used_specific_object ?sp ;
      crm:P177_assigned_property_of_type ?d .
  ?t a ktrans:Translation ; rdfs:label ?번역본 .
  ?p rdfs:label ?주체 . ?sp crm:P1_is_identified_by ?근거 .
  ?d rdfs:label ?차원 .
  OPTIONAL { ?cs crm:P141_assigned ?v . ?v rdfs:label ?값 . FILTER(lang(?값)="ko") }
  OPTIONAL { ?cs ktrans:hasPolarity ?극성 }
  FILTER(lang(?번역본)="ko" && lang(?주체)="ko" && lang(?차원)="ko")
} ORDER BY ?번역본 ?차원
""")
print(f"총 {len(stmt)}건")
stmt.head(20)

### 차원별 분포

분류표 10항목이 실제 비평에서 어떻게 쓰이는지 보여 준다.
충실도와 종결어미체가 가장 많고, 정확성·오역이력은 적다.

In [ ]:
dim = q("""
SELECT ?차원 (COUNT(?cs) AS ?건수)
WHERE {
  ?cs crm:P2_has_type kt:type-criticism ;
      crm:P177_assigned_property_of_type ?d .
  ?d rdfs:label ?차원 . FILTER(lang(?차원)="ko")
} GROUP BY ?차원 ORDER BY DESC(?건수)
""")
dim["건수"] = dim["건수"].astype(int)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(dim["차원"][::-1], dim["건수"][::-1], color="#6D2E46", alpha=.85, height=.65)
for i, v in enumerate(dim["건수"][::-1]):
    ax.text(v + .4, i, str(v), va="center", fontsize=9, color="#6D2E46")
ax.set_xlabel("진술 수")
ax.set_title("분류표 10항목의 실제 사용 분포", fontsize=12, color="#6D2E46", pad=12)
ax.set_yticks(range(len(dim)))
ax.set_yticklabels(dim["차원"][::-1], fontfamily=KFONT)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Q6. 상반된 평가의 공존

`E13 Attribute Assignment`는 어느 평가가 옳은지 판정하지 않고 **누가 그렇게 봤는지**만 기록한다.
1950년대 번역에 대한 부진론(조우호 등)과 그에 대한 반론(조향)이 충돌 없이 공존한다.
반론 진술은 대상이 번역본이 아니라 **다른 진술**이다.

In [ ]:
q("""
SELECT ?원진술 ?원진술자 ?극성1 ?반론 ?반론자 ?극성2 ?근거 ?rebuttal_en
WHERE {
  ?r crm:P2_has_type kt:st-rebuttal ; rdfs:label ?반론 , ?rebuttal_en ;
     crm:P14_carried_out_by ?rp ;
     crm:P140_assigned_attribute_to ?target ;
     crm:P16_used_specific_object ?sp .
  ?target rdfs:label ?원진술 ; crm:P14_carried_out_by ?tp .
  ?rp rdfs:label ?반론자 . ?tp rdfs:label ?원진술자 .
  ?sp crm:P1_is_identified_by ?근거 .
  OPTIONAL { ?target ktrans:hasPolarity ?극성1 }
  OPTIONAL { ?r ktrans:hasPolarity ?극성2 }
  FILTER(lang(?반론자)="ko" && lang(?원진술자)="ko")
  FILTER(lang(?반론)="ko" && lang(?원진술)="ko" && lang(?rebuttal_en)="en")
}
""")

## Q7. 저본유형 ≠ 작품유형 — 79건

UeDeKo의 '번역형태' 한 칸에는 성격이 다른 두 정보가 섞여 있었다.
두 축으로 나누자 저본유형과 작품유형이 어긋나는 79건이 드러난다.

대부분 **독일어에서 직접 옮긴 완역인데 전집·합본에 묶이며 편역으로 코딩된** 경우다.
번역의 성질이 아니라 출판 형태가 분류를 바꿨다.

In [ ]:
mismatch = q("""
SELECT ?번역본 ?연도 WHERE {
  ?t a ktrans:Translation ; rdfs:label ?번역본 ;
     crm:P3_has_note ?메모 ;
     crm:P4_has_time-span/crm:P82_at_some_time_within ?연도 .
  FILTER(CONTAINS(STR(?메모), "번역유형이 서로 다르다"))
  FILTER(lang(?번역본)="ko")
} ORDER BY ?연도
""")
print(f"총 {len(mismatch)}건")

mismatch["연대"] = (mismatch["연도"].astype(int)//10*10).astype(str) + "년대"
cnt = mismatch["연대"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(cnt.index, cnt.values, color="#6D2E46", alpha=.85, width=.6)
for x, v in zip(cnt.index, cnt.values):
    ax.text(x, v + .4, str(v), ha="center", fontsize=9, color="#6D2E46")
ax.set_ylabel("건수")
ax.set_title("저본유형 ≠ 작품유형 — 연대별 분포", fontsize=12, color="#6D2E46", pad=12)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Q8. 경유 정보의 공백 — 원 데이터의 상태

UeDeKo 원표에 '중역'은 12건 기재되어 있으나 **'직접'은 한 번도 기재되어 있지 않다.**
나머지 258건은 '직접'이 아니라 **미기재**다.

직접 번역을 기본값으로 두는 암묵적 전제가 데이터에 깔려 있었음이, 데이터로 만들고 나서야 드러난다.
그리고 메타데이터가 비어 있는 자리를 비평 텍스트가 메운다 — 저본이 특정·언급된 7건이 그것이다.

In [ ]:
via = q("""
SELECT ?경유 (COUNT(?t) AS ?건수) WHERE {
  ?t a ktrans:Translation ; ktrans:viaType ?v .
  ?v rdfs:label ?경유 . FILTER(lang(?경유)="ko")
} GROUP BY ?경유 ORDER BY DESC(?건수)
""")
via

## 정리

| 항목 | 값 |
|---|---|
| UeDeKo 레코드 | 318 |
| 작품 (전처리 후) | 271 |
| 저자 표기 이형 | 26 (+ 작중인물 1) |
| 역자 표기 → 정규형 | 155종 → 116명 |
| 저본유형 ≠ 작품유형 | 79 |
| 경유 '중역' 명시 | 12 (나머지 258은 미기재) |
| 저본 특정·언급 | 7 |
| 비평 진술 | 160 |
| 트리플 | 11,198 |

**이 연구의 범위**는 RDF 데이터 설계·편찬과 LOD 외부 연결(GND·VIAF·Wikidata·KAC)까지이며,
공개 서비스 구축은 후속 단계다.

**자체 정의는 세 개뿐**이고, 각각 출처가 다르다.
분량 축은 UeDeKo 편찬자가 판정해 둔 값을 계승했고,
경유 축은 한 칸에 섞여 있던 정보를 분리해 복원했으며,
극성은 줄글 비평을 질의 가능하게 만들기 위해 새로 부여했다.